# Exploratory Data Analysis and Visualization — UCI Adult Dataset

This notebook continues the Week 1 project using the same **UCI Adult (Census Income)** dataset.

Objectives:
- inspect the cleaned dataset structure;
- summarize numerical and categorical variables;
- visualize distributions and relationships;
- examine income-class imbalance;
- investigate education, workclass, occupation, marital status and sex in relation to income;
- inspect correlations among numerical variables;
- document the reasoning behind each finding.

The notebook is designed to be run from top to bottom so that the outputs can be captured as screenshots for the EDA report.

In [ ]:
# Install once if required
# !pip install ucimlrepo pandas numpy matplotlib seaborn scikit-learn

from ucimlrepo import fetch_ucirepo
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (9, 5)

adult = fetch_ucirepo(id=2)

X = adult.data.features.copy()
y = adult.data.targets.copy()
df = pd.concat([X, y], axis=1)

print("Dataset shape:", df.shape)
display(df.head())
display(df.info())

## 1. Basic EDA

In [ ]:
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

print("\nData types:")
display(df.dtypes.to_frame("dtype"))

print("\nDescriptive statistics:")
display(df.describe(include="all").T)

In [ ]:
# Missing-value audit before transformations
missing = df.isna().sum().sort_values(ascending=False)
missing_pct = (missing / len(df) * 100).round(2)

missing_report = pd.DataFrame({
    "missing_count": missing,
    "missing_percent": missing_pct
})

display(missing_report)

### Interpretation

The dataset contains 48,842 observations and 15 columns in the dataframe representation used in the notebook: 14 predictors and the `income` target. The numerical variables include age, final weight, education number, capital gain, capital loss and hours per week. The remaining predictors are categorical. The UCI documentation identifies `workclass`, `occupation`, and `native-country` as fields with missing values.

## 2. Target Distribution

In [ ]:
# Normalize target labels for plotting
plot_df = df.copy()
plot_df["income"] = plot_df["income"].astype(str).str.strip().str.replace(".", "", regex=False)

plt.figure(figsize=(7, 5))
ax = sns.countplot(data=plot_df, x="income")
plt.title("Income Class Distribution")
plt.xlabel("Income class")
plt.ylabel("Number of records")

for container in ax.containers:
    ax.bar_label(container, fmt="%d")

plt.tight_layout()
plt.show()

print(plot_df["income"].value_counts())

### Interpretation

The target is imbalanced: the `<=50K` class is substantially larger than the `>50K` class. This matters because a model can obtain deceptively high accuracy by favoring the majority class. For EDA, the imbalance is itself an important finding and should be reported before any predictive modeling is attempted.

## 3. Categorical Distributions

In [ ]:
categorical_cols = [
    "workclass", "education", "marital-status",
    "occupation", "relationship", "race", "sex"
]

for col in categorical_cols:
    print(f"\n{col}")
    display(plot_df[col].value_counts(dropna=False).to_frame("count").head(20))

In [ ]:
# Workclass
plt.figure(figsize=(10, 5))
order = plot_df["workclass"].value_counts().index
sns.countplot(data=plot_df, y="workclass", order=order)
plt.title("Workclass Distribution")
plt.xlabel("Number of records")
plt.ylabel("Workclass")
plt.tight_layout()
plt.show()

In [ ]:
# Education
plt.figure(figsize=(10, 6))
order = plot_df["education"].value_counts().index
sns.countplot(data=plot_df, y="education", order=order)
plt.title("Education Distribution")
plt.xlabel("Number of records")
plt.ylabel("Education")
plt.tight_layout()
plt.show()

In [ ]:
# Marital status
plt.figure(figsize=(10, 5))
order = plot_df["marital-status"].value_counts().index
sns.countplot(data=plot_df, y="marital-status", order=order)
plt.title("Marital Status Distribution")
plt.xlabel("Number of records")
plt.ylabel("Marital status")
plt.tight_layout()
plt.show()

In [ ]:
# Occupation
plt.figure(figsize=(10, 6))
order = plot_df["occupation"].value_counts().index
sns.countplot(data=plot_df, y="occupation", order=order)
plt.title("Occupation Distribution")
plt.xlabel("Number of records")
plt.ylabel("Occupation")
plt.tight_layout()
plt.show()

### Interpretation

The categorical distributions are not uniform. `Private` is the dominant workclass, while `HS-grad` and `Some-college` are among the most common education categories. Several occupation groups have similar frequencies, whereas a few categories are rare. These distributions are important because highly uneven categories can affect both visualization readability and downstream model learning.

## 4. Numerical Distributions

In [ ]:
numeric_cols = [
    "age", "fnlwgt", "education-num",
    "capital-gain", "capital-loss", "hours-per-week"
]

for col in numeric_cols:
    plt.figure(figsize=(8, 4.5))
    sns.histplot(data=plot_df, x=col, bins=30, kde=True)
    plt.title(f"Distribution of {col}")
    plt.xlabel(col)
    plt.ylabel("Frequency")
    plt.tight_layout()
    plt.show()

### Interpretation

Age is spread across the adult population, while `hours-per-week` is concentrated around the conventional full-time work range. `capital-gain` and `capital-loss` are highly right-skewed, with many observations at zero and a smaller number of large values. This explains why these variables can dominate simple scale-sensitive visualizations and why robust transformations are useful.

## 5. Income by Education

In [ ]:
education_income = pd.crosstab(
    plot_df["education"], plot_df["income"], normalize="index"
) * 100

education_income = education_income.sort_values(
    by=">50K", ascending=False
)

plt.figure(figsize=(10, 7))
education_income.plot(kind="barh", stacked=True, figsize=(10, 7))
plt.title("Income-Class Composition by Education")
plt.xlabel("Percentage of records")
plt.ylabel("Education")
plt.legend(title="Income")
plt.tight_layout()
plt.show()

display(education_income.round(2))

### Interpretation

The income composition changes noticeably across education levels. Higher educational attainment tends to be associated with a larger share of observations in the `>50K` group. This is an association, not proof that education alone causes higher income, because occupation, age, workclass and other variables are also related to income.

## 6. Income by Sex

In [ ]:
sex_income = pd.crosstab(
    plot_df["sex"], plot_df["income"], normalize="index"
) * 100

plt.figure(figsize=(8, 5))
sex_income.plot(kind="bar", figsize=(8, 5))
plt.title("Income-Class Composition by Sex")
plt.xlabel("Sex")
plt.ylabel("Percentage within sex")
plt.xticks(rotation=0)
plt.legend(title="Income")
plt.tight_layout()
plt.show()

display(sex_income.round(2))

### Interpretation

The proportion of observations above $50K differs substantially between the male and female groups in this historical dataset. This is an important EDA finding, but it should be interpreted cautiously: the dataset reflects 1994 U.S. census-derived information and contains demographic variables. Differences in the dataset should not be interpreted as causal explanations or as a justification for discriminatory decisions.

## 7. Income by Workclass and Occupation

In [ ]:
for col in ["workclass", "occupation"]:
    rate = pd.crosstab(
        plot_df[col], plot_df["income"], normalize="index"
    ) * 100

    rate = rate.sort_values(">50K", ascending=False)

    plt.figure(figsize=(10, 6))
    rate[" >50K" if " >50K" in rate.columns else ">50K"].plot(kind="barh")
    plt.title(f"Share of >50K Records by {col.replace('-', ' ').title()}")
    plt.xlabel("Percentage of records in category")
    plt.ylabel(col.replace("-", " ").title())
    plt.tight_layout()
    plt.show()

    display(rate.round(2))

## 8. Correlation Analysis

Pearson correlation is useful for examining linear relationships among numerical variables. It does not establish causation and is not a complete measure of association for categorical variables.

In [ ]:
corr = plot_df[numeric_cols].corr(numeric_only=True)

plt.figure(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation Matrix of Numerical Variables")
plt.tight_layout()
plt.show()

display(corr.round(2))

### Interpretation

The correlation matrix helps identify which numerical variables move together linearly. A relatively weak pairwise correlation does not mean a variable is unimportant for income classification, because income is categorical and relationships can be nonlinear or mediated by other features. The heatmap is therefore used as a diagnostic rather than as a feature-selection rule.

## 9. Hours Worked and Income

In [ ]:
plt.figure(figsize=(9, 5))
sns.boxplot(data=plot_df, x="income", y="hours-per-week")
plt.title("Hours per Week by Income Class")
plt.xlabel("Income class")
plt.ylabel("Hours per week")
plt.tight_layout()
plt.show()

display(
    plot_df.groupby("income")["hours-per-week"]
    .agg(["count", "mean", "median", "std", "min", "max"])
    .round(2)
)

### Interpretation

Comparing hours worked by income class helps assess whether workload differs between the two target groups. The boxplot should be interpreted together with the summary statistics because extreme values can stretch the vertical scale. Any observed difference is an association within this dataset rather than evidence that working more hours necessarily causes higher income.

## 10. Key EDA Findings

1. The dataset is strongly concentrated in the `<=50K` income class, so class imbalance is a major analytical consideration.
2. Workclass and education have visibly uneven category frequencies, with `Private` and `HS-grad`/`Some-college` among the most common groups.
3. Numerical variables such as capital gain and capital loss are highly skewed and contain many zero values.
4. Income composition varies across education levels, with higher education categories generally showing a larger share of `>50K` observations.
5. Income composition also differs by sex and across workclass/occupation categories, highlighting important associations that warrant cautious interpretation.
6. Pairwise numerical correlations are useful for understanding linear relationships, but they do not capture the full structure of a categorical income target.
7. EDA findings support the preprocessing choices from Week 1 and help identify which variables and relationships deserve attention in later modeling.

## 11. Conclusion

The EDA provided a structured view of the same Adult dataset used in the data-cleaning task. The analysis combined numerical summaries, categorical frequency analysis, distribution plots, grouped comparisons, boxplots and a correlation heatmap. The visualizations revealed class imbalance, skewed numerical variables and meaningful differences in income composition across several demographic and employment-related features. These observations provide a foundation for subsequent modeling while emphasizing that association should not be confused with causation.